# Manual Topic Validation Setup

This notebook prepares the materials for manual validation of the topic-to-dimension inference workflow. The goal is to create a clean, reproducible validation step that can be passed to a reviewer without requiring them to interact with the BERTopic model, embedding model, or any of the downstream prediction code.

The model assigns each non-outlier topic to one or more food access dimensions using cosine similarity between topic centroids and dimension definition embeddings. These similarities are converted to softmax probabilities, and dimension labels are assigned using a margin-based multilabel rule. Because the model now allows multilabel assignments, the manual validation process also needs to allow topics to map to more than one access dimension.

In the earlier validation workflow, manual rankings were assigned on a 1–5 scale, transformed into softmax-like values, and then converted into labels for comparison with model outputs. That approach was useful during development, but it required treating ordinal human rankings like model logits or probabilities. In this cleaned workflow, manual validation is simpler and more interpretable: the reviewer scores each dimension independently using a 0–3 relevance scale, and manual labels are derived directly from those scores.

The manual scoring rubric is:

- `0` = not represented
- `1` = weakly or ambiguously represented
- `2` = clearly represented
- `3` = strongly represented

These scores are not ranks. Multiple dimensions may receive the same score, which allows the manual validation process to reflect genuinely multidimensional topics. After scoring, the manual label set is derived using a strict top-dimension rule: all dimensions tied for the highest score are included as manual labels, provided the highest score is at least 2. Topics whose maximum score is 0 or 1 are treated as unclear because no dimension was confidently represented. Topics with four or more tied top-scoring dimensions are flagged as broad because they do not map cleanly onto a small set of access dimensions.

This notebook also separates topic validity from dimension scoring. Some modeled topics may be technically valid BERTopic clusters but not useful for downstream food-access analysis, such as topics dominated by generic reviews like “good,” “nice,” or “great store.” The Streamlit app therefore allows each topic to be marked as `valid`, `invalid`, or `needs_review`, with an optional reason for invalid topics. This makes topic exclusion explicit and reproducible rather than handled manually or implicitly.

To avoid biasing the manual reviewer, this notebook creates a blinded validation input file for the Streamlit app. The blinded file includes only topic metadata and example reviews, such as the topic name, OpenAI-generated label, keyword representation, review count, and selected representative reviews. It does not include model-predicted dimensions or model probabilities. A separate model reference file is saved for later comparison after manual scoring is complete.

For each topic, the validation file includes a hybrid set of review examples: the top reviews by topic assignment probability and a reproducible random sample of additional readable reviews. This gives the reviewer both central examples of the topic and some sense of within-topic variation. The displayed review text comes from the `text` column, which contains the original readable review text and includes translated text when applicable.

The overall workflow is:

1. Start from the final review-level dataframe with topic assignments, model predictions, topic metadata, and review text.
2. Exclude the BERTopic outlier topic (`Topic = -1`) from manual validation.
3. Build a topic-level validation table.
4. Select representative review examples for each topic.
5. Save a blinded input file for the Streamlit app.
6. Save a separate model reference file for later agreement analysis.
7. Use the Streamlit app to assign manual topic status and 0–3 dimension relevance scores.
8. Import the manual scores.
9. Derive manual label sets from the scores.
10. Compare manual labels with model-assigned labels using agreement and overlap metrics.

This structure keeps the validation process reproducible, blinded, and aligned with the model’s final multilabel assignment framework.

Control Panel with Handles for Changing Scoring:

In [3]:
# ============================================================
# Manual Validation Setup
# ============================================================
# Goal:
# Build blinded topic-level validation files for a Streamlit app.
#
# Workflow:
# 1. Use the final review-level dataframe (`reviews`) with topic assignments,
#    model predictions, topic metadata, and review text.
# 2. Exclude BERTopic outlier topic (-1) before manual validation.
# 3. Create a blinded input file for manual scoring.
# 4. Create a separate model reference file for later comparison.
# 5. Streamlit reads only the blinded file and writes manual scores.
#
# Manual scoring:
# 0 = not represented
# 1 = weakly or ambiguously represented
# 2 = clearly represented
# 3 = strongly represented
#
# Manual topic status:
# - valid: coherent/interpretable topic suitable for downstream analysis
# - invalid: too generic, incoherent, low-information, or not useful
# - needs_review: uncertain; revisit before final inclusion/exclusion
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

#import reviews df

reviews = pd.read_csv("data/reviews_with_dimensions.csv")


# -----------------------------
# Dimension order
# -----------------------------

DIM_ORDER = [
    "Availability",
    "Accessibility",
    "Accommodation",
    "Affordability",
    "Acceptability",
]


# -----------------------------
# Manual validation paths
# -----------------------------

VALIDATION_DIR = Path("data/manual_validation")
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

BLINDED_INPUT_PATH = VALIDATION_DIR / "manual_validation_input_blinded.csv"
MODEL_REFERENCE_PATH = VALIDATION_DIR / "manual_validation_model_reference.csv"
SCORES_OUTPUT_PATH = VALIDATION_DIR / "manual_validation_scores.csv"

APP_PATH = Path("apps/manual_validation_app.py")


# -----------------------------
# Manual scoring configuration
# -----------------------------

MANUAL_SCORE_MIN = 0
MANUAL_SCORE_MAX = 3

MANUAL_SCORE_RUBRIC = {
    0: "not represented",
    1: "weakly or ambiguously represented",
    2: "clearly represented",
    3: "strongly represented",
}

MANUAL_TOPIC_STATUS_OPTIONS = [
    "valid",
    "invalid",
    "needs_review",
]

MANUAL_INVALID_REASON_OPTIONS = [
    "too_generic",
    "too_short_or_low_information",
    "not_food_access_related",
    "mixed_or_incoherent",
    "duplicate_or_redundant",
    "other",
]


# -----------------------------
# Review example selection settings
# -----------------------------

N_TOP_REVIEWS = 3
N_RANDOM_REVIEWS = 3
MIN_REVIEW_CHARS = 20
RANDOM_SEED = 42


# -----------------------------
# Required columns check
# -----------------------------

required_review_cols = [
    "topic",
    "topic_prob_max",
    "text",
    "pred_dim",
    "pred_prob",
    "pred_dim2",
    "pred_prob2",
    "pred_dim3",
    "pred_prob3",
    "prob_margin",
    "Availability_p",
    "Accessibility_p",
    "Accommodation_p",
    "Affordability_p",
    "Acceptability_p",
    "Name",
    "Representation",
    "OpenAI",
]

missing_cols = [col for col in required_review_cols if col not in reviews.columns]

if missing_cols:
    raise ValueError(f"reviews is missing required columns: {missing_cols}")

print("Manual validation setup complete.")
print(f"Blinded input will be saved to: {BLINDED_INPUT_PATH}")
print(f"Model reference will be saved to: {MODEL_REFERENCE_PATH}")
print(f"Manual scores will be written by Streamlit to: {SCORES_OUTPUT_PATH}")

Manual validation setup complete.
Blinded input will be saved to: data\manual_validation\manual_validation_input_blinded.csv
Model reference will be saved to: data\manual_validation\manual_validation_model_reference.csv
Manual scores will be written by Streamlit to: data\manual_validation\manual_validation_scores.csv


Topic Reference Table

In [4]:
# ============================================================
# Prepare review text and validation universe
# ============================================================

reviews = reviews.copy()

# Use the raw readable review text for manual validation.
# - "text" is the unprocessed readable review text and includes translated text when needed.
# - "originalText" is only populated when the original review language was not English.
# - Therefore, "text" is used for manual validation display.
reviews["review_text_display"] = reviews["text"].fillna("").astype(str)

# Light whitespace cleanup for nicer display in the validation app
reviews["review_text_display"] = (
    reviews["review_text_display"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Create explicit integer Topic column for topic-level workflows
reviews["Topic"] = reviews["topic"].astype("Int64")

# Keep only reviews attached to topics that received model predictions.
# Topic = -1 is excluded because it is the BERTopic outlier cluster,
# not a coherent semantic topic for manual validation.
validation_reviews = reviews[
    reviews["Topic"].notna()
    & (reviews["Topic"] != -1)
    & reviews["pred_dim"].notna()
].copy()

validation_reviews["Topic"] = validation_reviews["Topic"].astype(int)

# Sanity checks
n_total_reviews = len(reviews)
n_topic_reviews = reviews["Topic"].notna().sum()
n_validation_reviews = len(validation_reviews)

n_total_topics = reviews["Topic"].dropna().nunique()
n_validation_topics = validation_reviews["Topic"].nunique()

excluded_topics = sorted(
    set(reviews["Topic"].dropna().astype(int).unique())
    - set(validation_reviews["Topic"].unique())
)

print("=== Manual validation universe ===")
print(f"Total reviews: {n_total_reviews:,}")
print(f"Reviews with topic assignment: {n_topic_reviews:,}")
print(f"Eligible validation reviews: {n_validation_reviews:,}")
print()
print(f"Total topics in reviews: {n_total_topics:,}")
print(f"Eligible validation topics: {n_validation_topics:,}")
print(f"Excluded topics: {excluded_topics}")

=== Manual validation universe ===
Total reviews: 35,779
Reviews with topic assignment: 35,731
Eligible validation reviews: 34,344

Total topics in reviews: 60
Eligible validation topics: 59
Excluded topics: [np.int64(-1)]


In [5]:
# ============================================================
# Build topic-level model reference table
# ============================================================

topic_model_cols = [
    "Topic",
    "Name",
    "Representation",
    "OpenAI",
    "pred_dim", "pred_prob",
    "pred_dim2", "pred_prob2",
    "pred_dim3", "pred_prob3",
    "prob_margin",
    "Availability_p",
    "Accessibility_p",
    "Accommodation_p",
    "Affordability_p",
    "Acceptability_p",
]

# One row per topic with model predictions and topic metadata
topic_reference = (
    validation_reviews[topic_model_cols]
    .drop_duplicates(subset=["Topic"])
    .sort_values("Topic")
    .reset_index(drop=True)
)

# Add number of reviews per topic
topic_counts = (
    validation_reviews
    .groupby("Topic")
    .size()
    .rename("n_reviews")
    .reset_index()
)

topic_reference = topic_reference.merge(
    topic_counts,
    on="Topic",
    how="left",
    validate="one_to_one"
)

# Reorder columns so count appears near the front
front_cols = ["Topic", "n_reviews", "Name", "Representation", "OpenAI"]
remaining_cols = [c for c in topic_reference.columns if c not in front_cols]

topic_reference = topic_reference[front_cols + remaining_cols]

print("=== Topic reference table ===")
print(f"Rows: {len(topic_reference):,}")
print(f"Unique topics: {topic_reference['Topic'].nunique():,}")


=== Topic reference table ===
Rows: 59
Unique topics: 59


Example Review Pull

In [6]:
# ============================================================
# Select representative review examples for each topic
# ============================================================

def select_topic_reviews(
    df: pd.DataFrame,
    topic_col: str = "Topic",
    text_col: str = "review_text_display",
    prob_col: str = "topic_prob_max",
    n_top: int = N_TOP_REVIEWS,
    n_random: int = N_RANDOM_REVIEWS,
    min_chars: int = MIN_REVIEW_CHARS,
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    """
    Select review examples for manual topic validation.

    For each topic, this returns:
    - top_review_1 ... top_review_n:
        reviews with the highest topic assignment probabilities
    - random_review_1 ... random_review_n:
        reproducible random readable reviews from the same topic,
        excluding the selected top reviews when possible

    The random sample is fixed using `random_state` so that the validation
    input file can be regenerated consistently.
    """
    rows = []

    for topic, g in df.groupby(topic_col):
        g = g.copy()

        # Keep readable, non-empty reviews
        readable = g[
            g[text_col].notna()
            & (g[text_col].astype(str).str.len() >= min_chars)
        ].copy()

        # Fallback if a topic has only very short reviews
        if readable.empty:
            readable = g[g[text_col].notna()].copy()

        # Deduplicate identical review text
        readable = readable.drop_duplicates(subset=[text_col])

        # Select top reviews by topic probability
        if prob_col in readable.columns:
            top = (
                readable
                .sort_values(prob_col, ascending=False)
                .head(n_top)
            )
        else:
            top = readable.head(n_top)

        # Select random reviews, excluding top reviews where possible
        remaining = readable.copy()

        if "reviewId" in remaining.columns and "reviewId" in top.columns:
            remaining = remaining[~remaining["reviewId"].isin(top["reviewId"])]
        else:
            remaining = remaining.drop(index=top.index, errors="ignore")

        if len(remaining) > 0:
            random_sample = remaining.sample(
                n=min(n_random, len(remaining)),
                random_state=random_state,
            )
        else:
            random_sample = remaining.head(0)

        out = {"Topic": int(topic)}

        # Store top reviews
        for i, txt in enumerate(top[text_col].tolist(), start=1):
            out[f"top_review_{i}"] = txt

        # Ensure fixed columns even if fewer reviews are available
        for i in range(1, n_top + 1):
            out.setdefault(f"top_review_{i}", "")

        # Store random reviews
        for i, txt in enumerate(random_sample[text_col].tolist(), start=1):
            out[f"random_review_{i}"] = txt

        # Ensure fixed columns even if fewer reviews are available
        for i in range(1, n_random + 1):
            out.setdefault(f"random_review_{i}", "")

        rows.append(out)

    return (
        pd.DataFrame(rows)
        .sort_values("Topic")
        .reset_index(drop=True)
    )


topic_review_examples = select_topic_reviews(validation_reviews)

print("=== Topic review examples ===")
print(f"Rows: {len(topic_review_examples):,}")
print(f"Unique topics: {topic_review_examples['Topic'].nunique():,}")

topic_review_examples.head()

=== Topic review examples ===
Rows: 59
Unique topics: 59


,Topic,top_review_1,top_review_2,top_review_3,random_review_1,random_review_2,random_review_3
0,0,It's a great to shop. Meats are fresh and grea...,Great place to find quality meat at a good price.,Good selection of meats. Most prices r reasona...,This would be the place to go for the good dea...,Clean great meat choices. Excellent prices.,"Variety of meats, produce and everyday needs. ..."
1,1,Great place to shop for groceries.,Great place to shop for groceries,Very good place to shop groceries!,Good grocery store. Cool atmosphere. Great sel...,Everything that I expected and more. I've foun...,Great place and prices for all your grocery ne...
2,2,Good experience at this location.,Always great experience with this location 😊,My first time in the place. It was very nice,Top quality experience,"So,so crowded all the time.",Pretty new and nice Harris Teeter. Good location.
3,3,Good prices on most everthing,Good prices on most things,Good prices for great stuff.,"Great prices, covers large spectrum of one's n...",I am happy. Every thing is nice and pretty. Ev...,Decent prices and products.
4,4,Great store with great prices,Great store but very expensive!!,Great store good prices!,Great prices & decent selection. Store was cle...,"Fantastic store to shop at, great prices.",Good stuff and really cheap. I always save mon...


In [7]:
# Spot-check one topic's selected examples
sample_topic = topic_review_examples.iloc[0]

print(f"Topic: {sample_topic['Topic']}")
print("\nTop reviews:")
for i in range(1, N_TOP_REVIEWS + 1):
    print(f"{i}. {sample_topic[f'top_review_{i}']}")

print("\nRandom reviews:")
for i in range(1, N_RANDOM_REVIEWS + 1):
    print(f"{i}. {sample_topic[f'random_review_{i}']}")

Topic: 0

Top reviews:
1. It's a great to shop. Meats are fresh and great prices.
2. Great place to find quality meat at a good price.
3. Good selection of meats. Most prices r reasonable. Very clean store and large selection.

Random reviews:
1. This would be the place to go for the good deals on meats and produce
2. Clean great meat choices. Excellent prices.
3. Variety of meats, produce and everyday needs. All reasonable priced. This store caters not only to family needs, but, the needs of a single person. #sproutsforthewin


Blinded App Input Table

In [8]:
# ============================================================
# Create blinded validation input and model reference files
# ============================================================

# Merge topic-level metadata/predictions with selected review examples
validation_topic_table = topic_reference.merge(
    topic_review_examples,
    on="Topic",
    how="left",
    validate="one_to_one",
)

# -----------------------------
# Blinded input for Streamlit
# -----------------------------
# This file should be used for manual grading.
# It intentionally excludes model-predicted dimensions and probabilities.

blinded_cols = [
    "Topic",
    "n_reviews",
    "Name",
    "Representation",
    "OpenAI",
]

review_example_cols = (
    [f"top_review_{i}" for i in range(1, N_TOP_REVIEWS + 1)]
    + [f"random_review_{i}" for i in range(1, N_RANDOM_REVIEWS + 1)]
)

blinded_input = validation_topic_table[
    blinded_cols + review_example_cols
].copy()


# -----------------------------
# Model reference for later comparison
# -----------------------------
# This file is NOT used by the Streamlit app.
# It contains model predictions/probabilities for agreement analysis
# after manual scoring is complete.

model_reference = validation_topic_table.copy()


# -----------------------------
# Save files
# -----------------------------

blinded_input.to_csv(BLINDED_INPUT_PATH, index=False)
model_reference.to_csv(MODEL_REFERENCE_PATH, index=False)

print("=== Saved manual validation files ===")
print(f"Blinded Streamlit input: {BLINDED_INPUT_PATH}")
print(f"Model reference file:    {MODEL_REFERENCE_PATH}")
print()
print(f"Blinded input rows:      {len(blinded_input):,}")
print(f"Model reference rows:    {len(model_reference):,}")
print(f"Unique topics:           {blinded_input['Topic'].nunique():,}")

=== Saved manual validation files ===
Blinded Streamlit input: data\manual_validation\manual_validation_input_blinded.csv
Model reference file:    data\manual_validation\manual_validation_model_reference.csv

Blinded input rows:      59
Model reference rows:    59
Unique topics:           59


File Verification to Make Sure app will work

In [9]:
# ============================================================
# Verify saved validation files
# ============================================================

blinded_check = pd.read_csv(BLINDED_INPUT_PATH)
model_ref_check = pd.read_csv(MODEL_REFERENCE_PATH)

model_prediction_cols = [
    "pred_dim", "pred_prob",
    "pred_dim2", "pred_prob2",
    "pred_dim3", "pred_prob3",
    "prob_margin",
    "Availability_p",
    "Accessibility_p",
    "Accommodation_p",
    "Affordability_p",
    "Acceptability_p",
]

leaked_cols = [col for col in model_prediction_cols if col in blinded_check.columns]

print("=== Validation file check ===")
print(f"Blinded input exists:       {BLINDED_INPUT_PATH.exists()}")
print(f"Model reference exists:     {MODEL_REFERENCE_PATH.exists()}")
print()
print(f"Blinded input rows:         {len(blinded_check):,}")
print(f"Model reference rows:       {len(model_ref_check):,}")
print()
print(f"Blinded unique topics:      {blinded_check['Topic'].nunique():,}")
print(f"Reference unique topics:    {model_ref_check['Topic'].nunique():,}")
print()
print(f"Model columns in blinded file: {leaked_cols}")

if leaked_cols:
    raise ValueError(
        "Blinded input contains model prediction columns. "
        "Remove these before using the file for manual validation."
    )

if len(blinded_check) != len(model_ref_check):
    raise ValueError("Blinded input and model reference have different row counts.")

if set(blinded_check["Topic"]) != set(model_ref_check["Topic"]):
    raise ValueError("Blinded input and model reference contain different topic IDs.")

print("Validation files look good.")

=== Validation file check ===
Blinded input exists:       True
Model reference exists:     True

Blinded input rows:         59
Model reference rows:       59

Blinded unique topics:      59
Reference unique topics:    59

Model columns in blinded file: []
Validation files look good.


## Directions for Running App:

## Running the Manual Topic Validation App

This project uses a small Streamlit app to manually validate topic labels. The app reads the blinded topic validation file and saves manual scores to a CSV.

The app should be run from the project root folder:

```text
Final_pipeline/
├── apps/
│   └── manual_validation_app.py
├── data/
│   └── manual_validation/
│       ├── manual_validation_input_blinded.csv
│       ├── manual_validation_model_reference.csv
│       └── manual_validation_scores.csv
```

The app reads:

```text
data/manual_validation/manual_validation_input_blinded.csv
```

and saves manual annotations to:

```text
data/manual_validation/manual_validation_scores.csv
```

The model reference file is **not** used by the app while scoring:

```text
data/manual_validation/manual_validation_model_reference.csv
```

That file is only used later, after manual scoring is complete, to compare manual labels against model predictions.

---

### Option 1: Run from Anaconda Prompt

Open **Anaconda Prompt**.

Activate the environment you want to use:

```bat
conda activate YOUR_ENV_NAME
```

For example:

```bat
conda activate thesis
```

Move to the project root folder. On Windows, if the project is on a different drive, use `cd /d`:

```bat
cd /d "G:\My Drive\Meeting Reports\Thesis\For_Coding\Final_pipeline"
```

Run the app:

```bat
python -m streamlit run apps/manual_validation_app.py
```

Streamlit should open the app in your browser automatically. If it does not, look for a local URL in the terminal, usually:

```text
http://localhost:8501
```

Copy and paste that URL into your browser.

---

### Option 2: Run from the terminal in VS Code or another IDE

Open the project folder in your IDE.

Open a new terminal.

Activate the correct environment, if needed:

```bat
conda activate YOUR_ENV_NAME
```

Make sure the terminal is in the project root folder. You should be in the folder that contains both `apps/` and `data/`.

You can check your current location with:

```bat
cd
```

or in PowerShell:

```powershell
Get-Location
```

If needed, move to the project root:

```bat
cd /d "G:\My Drive\Meeting Reports\Thesis\For_Coding\Final_pipeline"
```

Then run:

```bat
python -m streamlit run apps/manual_validation_app.py
```

---

### Important: Python Version

The Streamlit app requires **Python 3.10 or later**.

Check your Python version with:

```bat
python --version
```

If you see Python 3.10, 3.11, or 3.12, you should be fine.

If you see Python 3.9 or earlier, you may get an error like:

```text
TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'
```

This happens because older Python versions do not support newer type-hint syntax like:

```python
str | None
```

Use a different conda environment with Python 3.10+ if this happens.

---

### Check that Streamlit is installed

After activating your environment, check whether Streamlit is installed:

```bat
python -m streamlit --version
```

If Streamlit is not installed, install it with:

```bat
python -m pip install streamlit
```

Then rerun:

```bat
python -m streamlit run apps/manual_validation_app.py
```

---

### Common Troubleshooting

#### Problem: `streamlit` is not recognized

You may see:

```text
'streamlit' is not recognized as an internal or external command
```

or:

```text
streamlit : The term 'streamlit' is not recognized
```

Use:

```bat
python -m streamlit run apps/manual_validation_app.py
```

instead of:

```bat
streamlit run apps/manual_validation_app.py
```

This makes sure Streamlit runs from the currently active Python environment.

---

#### Problem: The terminal will not change to the `G:` drive

If you run:

```bat
cd "G:\My Drive\Meeting Reports\Thesis\For_Coding\Final_pipeline"
```

but the prompt still shows `C:\Users\...`, use:

```bat
cd /d "G:\My Drive\Meeting Reports\Thesis\For_Coding\Final_pipeline"
```

The `/d` tells Windows Command Prompt to change both the folder and the drive.

---

#### Problem: The app opens, but cannot find the input file

Make sure you launched the app from the project root folder, not from inside the `apps/` folder.

Correct:

```bat
cd /d "G:\My Drive\Meeting Reports\Thesis\For_Coding\Final_pipeline"
python -m streamlit run apps/manual_validation_app.py
```

Incorrect:

```bat
cd /d "G:\My Drive\Meeting Reports\Thesis\For_Coding\Final_pipeline\apps"
python -m streamlit run apps/manual_validation_app.py
```

If you are already inside the `apps/` folder, either go back to the project root:

```bat
cd ..
python -m streamlit run apps/manual_validation_app.py
```

or run the app using the local filename:

```bat
python -m streamlit run manual_validation_app.py
```

However, running from the project root is preferred because the app uses relative paths to find the validation files.

---

### Saving annotations

The app saves automatically when you click:

```text
💾 Save topic annotation
```

Each saved topic is written to:

```text
data/manual_validation/manual_validation_scores.csv
```

The browser download button is optional. You do not need to use it to save the CSV locally.

The output CSV only contains topics that have been saved. If you are testing the pipeline with a partially completed validation file, that is okay. The downstream notebook should handle missing topics and report how many topics have been scored so far.

App Code:

In [10]:
# ============================================================
# Write Streamlit manual validation app
# ============================================================

from pathlib import Path

APP_DIR = Path("apps")
APP_DIR.mkdir(parents=True, exist_ok=True)

APP_PATH = APP_DIR / "manual_validation_app.py"

app_code = r'''
# manual_validation_app.py
# Streamlit app for blinded manual topic validation
#
# Run from terminal:
# streamlit run apps/manual_validation_app.py

from pathlib import Path
from datetime import datetime

import pandas as pd
import streamlit as st


# ============================================================
# App configuration
# ============================================================

st.set_page_config(
    page_title="Manual Topic Validation",
    page_icon="🗂️",
    layout="wide",
)

DIM_ORDER = [
    "Availability",
    "Accessibility",
    "Accommodation",
    "Affordability",
    "Acceptability",
]

SCORE_RUBRIC = {
    0: "not represented",
    1: "weakly or ambiguously represented",
    2: "clearly represented",
    3: "strongly represented",
}

TOPIC_STATUS_OPTIONS = [
    "valid",
    "invalid",
    "needs_review",
]

INVALID_REASON_OPTIONS = [
    "",
    "too_generic",
    "too_short_or_low_information",
    "not_food_access_related",
    "mixed_or_incoherent",
    "duplicate_or_redundant",
    "other",
]

DEFAULT_INPUT_PATH = Path("data/manual_validation/manual_validation_input_blinded.csv")
DEFAULT_OUTPUT_PATH = Path("data/manual_validation/manual_validation_scores.csv")


# ============================================================
# Helpers
# ============================================================

@st.cache_data
def load_validation_input(path: str) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Could not find input file: {path}")

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    elif path.suffix.lower() in [".parquet", ".pq"]:
        df = pd.read_parquet(path)
    else:
        raise ValueError("Input file must be CSV or Parquet.")

    required_cols = [
        "Topic",
        "n_reviews",
        "Name",
        "Representation",
        "OpenAI",
    ]

    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Input file is missing required columns: {missing}")

    df["Topic"] = df["Topic"].astype(int)
    return df.sort_values("Topic").reset_index(drop=True)


def load_existing_scores(path: Path) -> pd.DataFrame:
    if path.exists():
        try:
            scores = pd.read_csv(path)
            if "Topic" in scores.columns:
                scores["Topic"] = scores["Topic"].astype(int)
                return scores
        except Exception:
            pass

    return pd.DataFrame(columns=[
        "Topic",
        "manual_topic_status",
        "manual_invalid_reason",
        *[f"manual_{d}" for d in DIM_ORDER],
        "manual_notes",
        "last_updated",
    ])


def save_scores(scores: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    scores = scores.sort_values("Topic").reset_index(drop=True)
    scores.to_csv(path, index=False)


def get_saved_row(scores: pd.DataFrame, topic: int) -> dict:
    if scores.empty or topic not in set(scores["Topic"]):
        return {}

    row = scores.loc[scores["Topic"] == topic].iloc[-1]
    return row.to_dict()


def upsert_score(scores: pd.DataFrame, new_row: dict) -> pd.DataFrame:
    topic = int(new_row["Topic"])

    if not scores.empty and topic in set(scores["Topic"]):
        scores = scores.loc[scores["Topic"] != topic].copy()

    scores = pd.concat([scores, pd.DataFrame([new_row])], ignore_index=True)
    return scores


def status_index(saved_value: str | None) -> int:
    if saved_value in TOPIC_STATUS_OPTIONS:
        return TOPIC_STATUS_OPTIONS.index(saved_value)
    return 0


def reason_index(saved_value: str | None) -> int:
    if saved_value in INVALID_REASON_OPTIONS:
        return INVALID_REASON_OPTIONS.index(saved_value)
    return 0


def score_default(saved_row: dict, dim: str) -> int:
    value = saved_row.get(f"manual_{dim}", 0)
    try:
        if pd.notna(value):
            return int(value)
    except Exception:
        pass
    return 0


def display_review_block(label: str, text: str) -> None:
    if pd.isna(text) or str(text).strip() == "":
        return

    st.markdown(f"**{label}**")
    st.info(str(text).strip())


# ============================================================
# Sidebar
# ============================================================

st.sidebar.header("Setup")

input_path = st.sidebar.text_input(
    "Blinded validation input path",
    value=str(DEFAULT_INPUT_PATH),
)

output_path = Path(
    st.sidebar.text_input(
        "Manual scores output path",
        value=str(DEFAULT_OUTPUT_PATH),
    )
)

st.sidebar.divider()

st.sidebar.header("Scoring rubric")
for score, description in SCORE_RUBRIC.items():
    st.sidebar.markdown(f"**{score}** = {description}")

st.sidebar.divider()

st.sidebar.caption(
    "Manual labels will be derived later from the highest-scoring dimension(s). "
    "Do not use model predictions while scoring."
)


# ============================================================
# Load data
# ============================================================

try:
    df = load_validation_input(input_path)
except Exception as e:
    st.error(f"Could not load validation input: {e}")
    st.stop()

scores = load_existing_scores(output_path)

completed_topics = set(scores.loc[scores["manual_topic_status"].notna(), "Topic"]) if not scores.empty else set()
n_completed = len(completed_topics)


# ============================================================
# Session state
# ============================================================

if "row_idx" not in st.session_state:
    st.session_state.row_idx = 0


# ============================================================
# Header
# ============================================================

st.title("🗂️ Manual Topic Validation")
st.caption(
    "Review each topic using blinded topic metadata and example reviews. "
    "Assign topic validity and 0–3 relevance scores for each access dimension."
)

progress = n_completed / len(df) if len(df) else 0
st.progress(progress, text=f"{n_completed} of {len(df)} topics saved")


# ============================================================
# Navigation
# ============================================================

col_prev, col_mid, col_next = st.columns([1, 2, 1])

with col_prev:
    if st.button("⬅️ Previous", use_container_width=True):
        st.session_state.row_idx = max(0, st.session_state.row_idx - 1)

with col_mid:
    st.markdown(
        f"<div style='text-align:center; font-weight:600;'>"
        f"Topic {st.session_state.row_idx + 1} of {len(df)}"
        f"</div>",
        unsafe_allow_html=True,
    )

with col_next:
    if st.button("Next ➡️", use_container_width=True):
        st.session_state.row_idx = min(len(df) - 1, st.session_state.row_idx + 1)

topic_ids = df["Topic"].tolist()
current_topic = int(df.iloc[st.session_state.row_idx]["Topic"])

jump_topic = st.selectbox(
    "Jump to Topic ID",
    options=topic_ids,
    index=topic_ids.index(current_topic),
)

if jump_topic != current_topic:
    st.session_state.row_idx = topic_ids.index(jump_topic)
    current_topic = int(jump_topic)

row = df.iloc[st.session_state.row_idx]
saved_row = get_saved_row(scores, current_topic)


# ============================================================
# Topic display
# ============================================================

st.divider()

left, right = st.columns([1, 2])

with left:
    st.subheader(f"Topic {int(row['Topic'])}")
    st.metric("Number of reviews", int(row["n_reviews"]))

    st.markdown("**Name**")
    st.write(row.get("Name", ""))

    st.markdown("**OpenAI label**")
    st.write(row.get("OpenAI", ""))

    st.markdown("**Representation / keywords**")
    st.write(row.get("Representation", ""))

with right:
    st.subheader("Example reviews")

    with st.expander("Top reviews by topic probability", expanded=True):
        for i in range(1, 4):
            display_review_block(
                f"Top review {i}",
                row.get(f"top_review_{i}", ""),
            )

    with st.expander("Random readable reviews", expanded=True):
        for i in range(1, 4):
            display_review_block(
                f"Random review {i}",
                row.get(f"random_review_{i}", ""),
            )


# ============================================================
# Manual annotation form
# ============================================================

st.divider()
st.subheader("Manual annotation")

with st.form(key=f"manual_form_topic_{current_topic}"):
    status = st.radio(
        "Topic status",
        options=TOPIC_STATUS_OPTIONS,
        index=status_index(saved_row.get("manual_topic_status")),
        horizontal=True,
        help=(
            "Mark invalid if the topic is too generic, incoherent, "
            "low-information, or not useful for downstream food-access analysis."
        ),
    )

    invalid_reason = ""

    if status == "invalid":
        invalid_reason = st.selectbox(
            "Invalid reason",
            options=INVALID_REASON_OPTIONS,
            index=reason_index(saved_row.get("manual_invalid_reason")),
        )

        if invalid_reason == "":
            st.warning("Please select an invalid reason before saving.")

    else:
        invalid_reason = ""

    st.markdown("### Dimension relevance scores")

    st.caption(
        "Score each dimension independently. These are not ranks; multiple dimensions may receive the same score."
    )

    score_cols = st.columns(len(DIM_ORDER))
    manual_scores = {}

    for i, dim in enumerate(DIM_ORDER):
        with score_cols[i]:
            manual_scores[dim] = st.radio(
                dim,
                options=[0, 1, 2, 3],
                index=score_default(saved_row, dim),
                format_func=lambda x: f"{x} — {SCORE_RUBRIC[x]}",
                key=f"score_{current_topic}_{dim}",
            )

    manual_notes = st.text_area(
        "Notes",
        value=str(saved_row.get("manual_notes", "")) if pd.notna(saved_row.get("manual_notes", "")) else "",
        placeholder="Optional notes about ambiguity, invalidity, broad topics, or scoring rationale.",
    )

    submitted = st.form_submit_button("💾 Save topic annotation", type="primary")

if submitted:
    if status == "invalid" and invalid_reason == "":
        st.error("Invalid topics need an invalid reason before saving.")
    else:
        new_row = {
            "Topic": current_topic,
            "manual_topic_status": status,
            "manual_invalid_reason": invalid_reason,
            **{f"manual_{dim}": int(score) for dim, score in manual_scores.items()},
            "manual_notes": manual_notes,
            "last_updated": datetime.now().isoformat(timespec="seconds"),
        }

        scores = upsert_score(scores, new_row)
        save_scores(scores, output_path)

        st.success(f"Saved annotation for Topic {current_topic} to {output_path}")

        # Move to next topic after save unless already at end
        if st.session_state.row_idx < len(df) - 1:
            st.session_state.row_idx += 1
            st.rerun()


# ============================================================
# Preview saved scores
# ============================================================

st.divider()

with st.expander("Preview saved annotations"):
    if output_path.exists():
        saved_preview = pd.read_csv(output_path)
        st.dataframe(saved_preview, use_container_width=True)
        st.download_button(
            "Download manual scores CSV",
            data=saved_preview.to_csv(index=False),
            file_name=output_path.name,
            mime="text/csv",
        )
    else:
        st.caption("No saved annotations yet.")
'''

APP_PATH.write_text(app_code, encoding="utf-8")

print(f"Wrote Streamlit app to: {APP_PATH}")
print()
print("Run this from your terminal:")
print(f"streamlit run {APP_PATH}")

Wrote Streamlit app to: apps\manual_validation_app.py

Run this from your terminal:
streamlit run apps\manual_validation_app.py


In [11]:
print("Manual validation app is ready.")
print()
print("From your project root, run:")
print(f"streamlit run {APP_PATH}")
print()
print("The app will read:")
print(BLINDED_INPUT_PATH)
print()
print("The app will save manual scores to:")
print(SCORES_OUTPUT_PATH)

Manual validation app is ready.

From your project root, run:
streamlit run apps\manual_validation_app.py

The app will read:
data\manual_validation\manual_validation_input_blinded.csv

The app will save manual scores to:
data\manual_validation\manual_validation_scores.csv


Final Check 


In [12]:
# ============================================================
# Final handoff check
# ============================================================

print("=== Manual validation setup handoff check ===")
print()

# Required setup files
setup_files = {
    "Blinded validation input": BLINDED_INPUT_PATH,
    "Model reference file": MODEL_REFERENCE_PATH,
    "Streamlit app": APP_PATH,
}

for label, path in setup_files.items():
    exists = path.exists()
    print(f"{label}: {'FOUND' if exists else 'MISSING'}")
    print(f"  {path}")

print()

# Check whether manual scoring has started
if SCORES_OUTPUT_PATH.exists():
    manual_scores_preview = pd.read_csv(SCORES_OUTPUT_PATH)

    n_scored = manual_scores_preview["Topic"].nunique()
    n_total = pd.read_csv(BLINDED_INPUT_PATH)["Topic"].nunique()
    n_remaining = n_total - n_scored

    print("Manual scores file: FOUND")
    print(f"  {SCORES_OUTPUT_PATH}")
    print()
    print(f"Topics available for validation: {n_total:,}")
    print(f"Topics scored so far:            {n_scored:,}")
    print(f"Topics remaining:                {n_remaining:,}")

    if "manual_topic_status" in manual_scores_preview.columns:
        print()
        print("Topic status counts:")
        print(manual_scores_preview["manual_topic_status"].value_counts(dropna=False))

else:
    n_total = pd.read_csv(BLINDED_INPUT_PATH)["Topic"].nunique()

    print("Manual scores file: NOT FOUND YET")
    print(f"  Expected path: {SCORES_OUTPUT_PATH}")
    print()
    print(f"Topics available for validation: {n_total:,}")
    print("Topics scored so far:            0")
    print()
    print("Run the Streamlit app and save at least one topic annotation to create this file.")



=== Manual validation setup handoff check ===

Blinded validation input: FOUND
  data\manual_validation\manual_validation_input_blinded.csv
Model reference file: FOUND
  data\manual_validation\manual_validation_model_reference.csv
Streamlit app: FOUND
  apps\manual_validation_app.py

Manual scores file: FOUND
  data\manual_validation\manual_validation_scores.csv

Topics available for validation: 59
Topics scored so far:            15
Topics remaining:                44

Topic status counts:
manual_topic_status
valid      9
invalid    6
Name: count, dtype: int64
